# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chay tu tren xuong. Runtime > Change runtime type > **T4 GPU** truoc khi bat dau.

| O | Lam gi | Thoi gian |
|---|---|---|
| 1 | clone + install | ~1 phut |
| 2 | smoke: import + unit test | ~30 giay |
| 3 | **core pipeline NB1 -> NB5** | ~80 phut |
| 4 | gatekeeper + in ket qua | ~10 giay |


In [ ]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

REPO = "https://github.com/VinUni-AI20k/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/content/Day21-Track3-Finetuning-Lab")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


In [ ]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke


In [ ]:
# @title 3. Core pipeline — NB1 → NB5
# EVAL_LIMIT truncates both eval sets: "" = full run (submittable),
# 8 = ~fast smoke pass. STAGES lets you resume after a failure.
import os
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = ""         # @param ["", "4", "8", "16", "25"]
STAGES       = "nb4 nb5"   # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


In [ ]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null


## Sao lưu kết quả trước khi kết thúc runtime
Gói giữ cả ba adapter đối chứng để không mất công train. Kết quả smoke chưa phải bài nộp đầy đủ.


In [ ]:
# @title 5. Sao lưu ZIP sang Google Drive (không train lại)
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
import shutil

# Hỗ trợ cả RUN_ALL gốc và bản dùng workspace riêng.
artifact_root = Path(ROOT).resolve() if "ROOT" in globals() else Path("/content/Day21-Track3-Finetuning-Lab")
assert (artifact_root / "results").is_dir(), f"Không tìm thấy results tại {artifact_root}"
archive_path = artifact_root.parent / ("lab21_" + artifact_root.name + "_backup.zip")
with ZipFile(archive_path, "w", ZIP_DEFLATED) as archive:
    for folder in ["results", "data/split", "adapters/correct", "adapters/attn_only", "adapters/wrong_lr", "adapters/qlora", "submission", "notebooks", "run_all_support"]:
        directory = artifact_root / folder
        if directory.exists():
            for artifact in sorted(directory.rglob("*")):
                if artifact.is_file() and "__pycache__" not in artifact.parts and artifact.name != ".gitkeep":
                    archive.write(artifact, artifact.relative_to(artifact_root))
    for name in ["requirements.txt", "requirements-cpu.txt", "pyproject.toml"]:
        if (artifact_root / name).is_file(): archive.write(artifact_root / name, name)
with ZipFile(archive_path) as archive:
    assert archive.testzip() is None, "ZIP bị lỗi"
print("ZIP an toàn trên runtime:", archive_path)

from google.colab import drive
mount_root = Path("/content/drive")
try:
    if not (mount_root / "MyDrive").is_dir():
        drive.mount(str(mount_root))
    assert (mount_root / "MyDrive").is_dir(), "Drive chưa được mount."
    destination_dir = mount_root / "MyDrive" / "Lab21_Backups"
    destination_dir.mkdir(parents=True, exist_ok=True)
    from datetime import datetime, timezone
    destination = destination_dir / (datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f_") + archive_path.name)
    shutil.copy2(archive_path, destination)
    assert destination.stat().st_size == archive_path.stat().st_size
    print("ĐÃ LƯU TRÊN GOOGLE DRIVE:", destination)
    print("Mở Google Drive > My Drive > Lab21_Backups, tải ZIP về máy.")
except Exception as exc:
    print(f"CHƯA LƯU LÊN DRIVE: {type(exc).__name__}: {exc}")
    print("VS Code có thể không hỗ trợ xác thực drive.mount. ZIP vẫn ở đường dẫn trên; đừng xóa runtime.")
